## RTB Practice - Inverse Kinematics Warm-up

### 0 - Setup
Loads the toy arm and opens Swift inside the notebook, the same way as Lab 2.
If you can see the arm under this cell, Swift is ready for Lab 2

In [11]:
import time
import numpy as np
from pathlib import Path
from roboticstoolbox import Robot
from spatialmath import SE3
import spatialgeometry as sg
import swift

HERE = Path.cwd().as_posix()
if hasattr(Robot, 'URDF_read'):             # older roboticstoolbox
    links, name = Robot.URDF_read('toy_robot/robot.urdf', tld=HERE)[:2]
else:                                       # roboticstoolbox 1.5 and newer
    from roboticstoolbox.models.URDF import URDFRobot
    from xacrodoc.packages import update_package_cache
    update_package_cache({'toy_robot': HERE + '/toy_robot'})
    links, name = URDFRobot.URDF_read(HERE + '/toy_robot/robot.urdf')[:2]
robot = Robot(links, name=name)
print('loaded', name, robot.n, 'joints')

env = swift.Swift()
env.launch(realtime=True, browser="notebook")
env.add(robot)
robot.q = [0.0, 0.0]   # home configuration
env.step()

loaded toy_2link 2 joints


/tmp/ipykernel_63470/205133883.py:21: FutureWarning: launch(browser="notebook") is deprecated, use launch(notebook=True)
  env.launch(realtime=True, browser="notebook")


In [2]:
# Syntax (reference only - do not run as-is):
# robot.q = q ; env.step()            # move the arm in Swift to joint values q
# robot.fkine(q).t                    # forward kinematics -> tip position [x, y, z]
# np.linalg.inv(T)                    # inverse of a 4x4 transform
# np.append(v, 1)                     # [x, y, z] -> [x, y, z, 1]
# v[:3]                               # first three entries of v
# np.linalg.norm(a - b)               # distance between two points
# ball = sg.Sphere(radius=0.03, color=(1.0, 0.1, 0.1, 0.6))   # red ball
# ball.T = SE3(*P) ; env.add(ball) ; env.step()               # put it at point P = [x, y, z]
# robot.ikine_LM(SE3(*P), q0=..., mask=[1,1,0,0,0,0])         # numerical IK (match x, y) -> sol
# sol.q , sol.success                 # the joint solution, did it converge

### 1 - The toy 2-link planar arm
Same arm as Workshop 0. Link 1: length 0.15 m. Link 2: length 0.20 m. Both revolute, planar, 0.05 m above the floor.
At q = [0, 0] the arm is straight along x. Move it to `q_bent` and watch Swift

In [12]:
print('tip at q = [0, 0]       :', np.round(robot.fkine([0, 0]).t, 4))

q_bent = np.array([np.deg2rad(30), np.deg2rad(45)])
# edit here: move the arm to q_bent
robot.q = q_bent
env.step()

print('tip at q = [30, 45] deg :', np.round(robot.fkine(q_bent).t, 4))

tip at q = [0, 0]       : [0.35 0.   0.05]
tip at q = [30, 45] deg : [0.1817 0.2682 0.05  ]


### 2 - Camera frame to base frame
Copy `M` and `T` from the README. The camera sees the object at `M_cam` (camera frame). `T` is the base frame seen from the camera.
Find the object position in the base frame (`Pobj`) with the inverse of `T`, also on paper, then put a red ball there in Swift

In [8]:
import numpy as np
from spatialmath import SE3

M_cam = (-0.0770,  0.1833,  0.9006)
T = np.array([
    [   0.9923,   0.0348,   0.1192,    0.0490 ],
    [   0.0546,  -0.9844,  -0.1673,    0.0364 ],
    [   0.1115,   0.1725,  -0.9787,    1.1965 ],
    [   0.0000,   0.0000,   0.0000,    1.0000 ],
])
Pobj = (SE3(np.linalg.inv(T))*SE3(M_cam)).t                               # edit here: object position in the base frame, [x, y, z]
hand_calc = np.array([0.0, 0.0, 0.0])     # edit here: your answer on paper

if Pobj is None:
    print('fill in Pobj')
else:
    print('Pobj     :', np.round(Pobj, 4))
    print('hand calc:', np.round(hand_calc, 4))
    print('diff     :', round(float(np.linalg.norm(Pobj - hand_calc)), 4), 'm')

# edit here: put a red ball at Pobj

ValueError: bad argument to constructor

### 3 - Let the toolbox solve IK
Solve IK for `Pobj` with `ikine_LM`. `mask=[1,1,0,0,0,0]` means match x and y only.
Then move the arm to the solution: the tip should land on the red ball

In [ ]:
from spatialmath import SE3

sol = robot.ikine_LM(SE3(Pobj), q0=[0, 0], mask=[1, 1, 0, 0, 0, 0]) # edit here: ikine_LM for Pobj, start from q0=[0, 0]

if sol is None:
    print('fill in the ikine_LM call')
else:
    print('q   :', np.round(sol.q, 4), '| ok', sol.success)
    print('tip :', np.round(robot.fkine(sol.q).t, 4))
    
# edit here: move the arm to sol.q
robot.q = sol.q
env.step()

q   : [-0.2836  1.5741] | ok True
tip : [0.1993 0.1502 0.05  ]


### 4 - Solve IK by hand and compare
The arm can reach `Pobj` with the elbow bent either way, so there are 2 branches. Work out both on paper,
then check each one with FK and show it in Swift

In [ ]:
# edit here: your hand-calculated (q1, q2) for each branch, in rad
hand_branches = [
    np.array([-0.2838,  np.pi/2]),   # branch 1 (elbow +90°)
    np.array([ np.pi/2, -np.pi/2]),  # branch 2 (elbow -90°)
]

for i, qb in enumerate(hand_branches):
    tip = robot.fkine(qb).t    # edit here: tip position of qb from FK
    err = np.linalg.norm(tip-Pobj)    # edit here: distance from tip to Pobj
    if err is None:
        print('fill in tip and err')
        break
    print('branch', i + 1, np.round(qb, 4), '-> tip', np.round(tip, 4), '| err', round(float(err), 4), 'm')

    # edit here: move the arm to qb
    robot.q = qb
    env.step()

    time.sleep(2)

branch 1 [-0.2838  1.5708] -> tip [0.2  0.15 0.05] | err 0.0 m
branch 2 [ 1.5708 -1.5708] -> tip [0.2  0.15 0.05] | err 0.0 m


### 5 - Change the start guess and see which branch the toolbox finds
Solve IK again from each `q0` and print the solution each time.

In [ ]:
for q0 in ([0, 0], [0, 1], [0, -1], [1, -1]):
    sol = robot.ikine_LM(SE3(Pobj),q0=q0,mask=[1,1,0,0,0,0])    # edit here: ikine_LM for Pobj, start from q0
    if sol is None:
        print('fill in the ikine_LM call')
        break
    print(f'q0={q0} -> q {np.round(sol.q, 4)}')

    robot.q = q0
    env.step
    time.sleep(2)

    robot.q = sol.q
    env.step()
    time.sleep(2)

q0=[0, 0] -> q [-0.2836  1.5741]
q0=[0, 1] -> q [-0.2839  1.5712]
q0=[0, -1] -> q [ 1.5698 -1.5757]
q0=[1, -1] -> q [ 1.5708 -1.5708]


### 6 - Your own robot
Put your LAB1 robot in `my_robot/` and run `python check_robot.py` first (README, Robot Setup).
This loads it exactly like the Lab 2 Setup cell, in a second Swift view under this cell

In [ ]:
my_robot = None
if not Path('my_robot/robot.urdf').exists():
    print('put your LAB1 robot in my_robot/ first (README, Robot Setup)')
else:
    if hasattr(Robot, 'URDF_read'):             # older roboticstoolbox
        links, name = Robot.URDF_read('my_robot/robot.urdf', tld=HERE)[:2]
    else:                                       # roboticstoolbox 1.5 and newer
        from roboticstoolbox.models.URDF import URDFRobot
        from xacrodoc.packages import update_package_cache
        update_package_cache({'my_robot': HERE + '/my_robot'})
        links, name = URDFRobot.URDF_read(HERE + '/my_robot/robot.urdf')[:2]
    my_robot = Robot(links, name=name)
    print('loaded', name, my_robot.n, 'joints')
    print('joint limits, row 1 = lower, row 2 = upper (rad for R, m for P):')
    print(np.round(my_robot.qlim, 4))

    env2 = swift.Swift()
    env2.launch(realtime=True, browser="notebook")
    env2.add(my_robot)
    my_robot.q = np.zeros(my_robot.n)   # home configuration
    env2.step()

loaded robot 3 joints
joint limits, row 1 = lower, row 2 = upper (rad for R, m for P):
[[-3.1416 -3.1416  0.    ]
 [ 3.1416  3.1416  0.3   ]]


/tmp/ipykernel_65513/3779202525.py:18: FutureWarning: launch(browser="notebook") is deprecated, use launch(notebook=True)
  env2.launch(realtime=True, browser="notebook")


Pick joint values inside the joint limits and move your robot there. Check the tip position against your MDH model from LAB1

In [ ]:

"""""
Joint Test:  [0.52359878 0.52359878 0.18      ]
MDH  tip pos : [0.2713 0.3875 0.8875]
URDF tip pos : [0.2712 0.3875 0.8875]
pos diff     : 0.0 m
-----------------------------------
MDH  tip R   :
 [[-0.433  0.5    0.75 ]
 [-0.25  -0.866  0.433]
 [ 0.866  0.     0.5  ]]
URDF tip R   :
 [[-0.433  0.5    0.75 ]
 [-0.25  -0.866  0.433]
 [ 0.866  0.     0.5  ]]
R diff       : 0.0
----------------------------------- 

Joint Test:  [-0.43633231 -0.43633231  0.135     ]
MDH  tip pos : [0.4542 0.0089 0.4498]
URDF tip pos : [0.4542 0.0089 0.4498]
pos diff     : 0.0 m
-----------------------------------
MDH  tip R   :
 [[ 0.383  -0.4226  0.8214]
 [-0.1786 -0.9063 -0.383 ]
 [ 0.9063  0.     -0.4226]]
URDF tip R   :
 [[ 0.383  -0.4226  0.8214]
 [-0.1786 -0.9063 -0.383 ]
 [ 0.9063 -0.     -0.4226]]
R diff       : 0.0
-----------------------------------
"""""
joint_configutations = [[0.52359878 ,0.52359878 ,0.18      ],[-0.43633231 ,-0.43633231  ,0.135     ]]
ball1 = sg.Sphere(radius=0.02, pose=SE3(0.2713, 0.3875, 0.8875), color=[1.0, 0.0, 0.0, 1.0])
ball2 = sg.Sphere(radius=0.02, pose=SE3(0.4542, 0.0089, 0.4498), color=[0.0, 0.0, 1.0, 1.0])
env2.add(ball1)
env2.add(ball2)
env2.step()
if my_robot is None:
    print('load your robot in the cell above first')
else:
    for q_try in joint_configutations:
        my_robot.q = q_try
        tip = robot.fkine(q_try).t
        env2.step()

        time.sleep(2)
    
        print('tip at q_try:', np.round(my_robot.fkine(q_try).t, 4))

tip at q_try: [0.2712 0.3875 0.8875]
tip at q_try: [0.4542 0.0089 0.4498]
